<a href="https://colab.research.google.com/github/visionbyangelic/Brain-Aging/blob/main/data/OASIS-3%20feature%20inventory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Brain-Age Prediction — Build 1
## Stage 0: OASIS-3 Feature Inventory & Cross-Dataset Compatibility

### Objective

This notebook documents and prepares the **OASIS-3 feature space** for Build 1 of the brain-age prediction project.

The immediate goal is to establish the available OASIS-3 morphometric features and determine their compatibility with the available OpenBHB features before developing the normative brain-age model.

### Build 1 Scientific Plan

1. Establish the OASIS-3 feature inventory.
2. Inspect and document the available OpenBHB feature space.
3. Determine whether the datasets have compatible features.
4. Select the appropriate feature strategy:
   - **Option A:** Reduced shared features
   - **Option B:** Regional feature parity
5. Train the normative brain-age model using healthy OpenBHB participants.
6. Freeze the model and evaluate it on OASIS-3.
7. Calculate brain-age gap (BAG).
8. Investigate associations between BAG and clinical status in OASIS-3.

### Stage 0 Scope

This notebook focuses on **feature inventory and cross-dataset compatibility**.

No brain-age model will be trained until feature compatibility between OASIS-3 and OpenBHB has been established.

### Scientific Rules

- Unit of analysis: **participant**
- Age prediction uses **morphometric MRI features only**
- Clinical and diagnostic labels are not inputs to the normative age model
- Feature definitions, names, and units must be documented before cross-dataset modelling
- The final OpenBHB → OASIS-3 model will be evaluated using a frozen model
- Brain-age gap is an analytical marker and **not a clinical diagnosis**

### Current Data Status

- **OASIS-3:** Available
- **OpenBHB:** Available
- **African/Nigerian MRI data:** Not currently available; reserved for a later build

---

**Build:** 1  
**Stage:** 0 — Feature Inventory & Compatibility  
**Status:** In progress

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ============================================================
# Stage 0 — OASIS-3 Feature Inventory
# Environment and paths
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# Project root
ROOT = Path("/content/drive/MyDrive/ANR_BrainAge")

# OASIS-3 data and manifest directories
DATA = ROOT / "data" / "oasis3"
MAN = ROOT / "manifests" / "oasis3"

# Existing OASIS-3 feature matrix
FEATURE_MATRIX = MAN / "OASIS3_feature_matrix.csv"

print("Environment initialized.")
print(f"Project root: {ROOT}")
print(f"OASIS-3 data: {DATA}")
print(f"OASIS-3 manifests: {MAN}")
print(f"Feature matrix: {FEATURE_MATRIX}")
print(f"File exists: {FEATURE_MATRIX.exists()}")

Environment initialized.
Project root: /content/drive/MyDrive/ANR_BrainAge
OASIS-3 data: /content/drive/MyDrive/ANR_BrainAge/data/oasis3
OASIS-3 manifests: /content/drive/MyDrive/ANR_BrainAge/manifests/oasis3
Feature matrix: /content/drive/MyDrive/ANR_BrainAge/manifests/oasis3/OASIS3_feature_matrix.csv
File exists: True


In [3]:
# ============================================================
# Stage 0 — Load OASIS-3 Feature Matrix
# ============================================================

oasis = pd.read_csv(FEATURE_MATRIX)

print("OASIS-3 feature matrix loaded.")
print(f"Shape: {oasis.shape}")
print(f"Unique participants: {oasis['Subject'].nunique()}")

print("\nColumns:")
print(oasis.columns.tolist())

OASIS-3 feature matrix loaded.
Shape: (1049, 210)
Unique participants: 1049

Columns:
['Subject', 'MR_session', 'FS_FSDATA ID', 'Subject_accession', 'Freesurfer_accession', 'FS QC Status', 'version', 'IntraCranialVol', 'lhCortexVol', 'rhCortexVol', 'CortexVol', 'SubCortGrayVol', 'TotalGrayVol', 'SupraTentorialVol', 'lhCorticalWhiteMatterVol', 'rhCorticalWhiteMatterVol', 'CorticalWhiteMatterVol', '3rd-Ventricle_volume', '4th-Ventricle_volume', '5th-Ventricle_volume', 'Brain-Stem_volume', 'CC_Anterior_volume', 'CC_Central_volume', 'CC_Mid_Anterior_volume', 'CC_Mid_Posterior_volume', 'CC_Posterior_volume', 'CSF_volume', 'Left-Accumbens-area_volume', 'Left-Amygdala_volume', 'Left-Caudate_volume', 'Left-Cerebellum-White-Matter_volume', 'Left-Cerebellum-Cortex_volume', 'Left-choroid-plexus_volume', 'Left-Hippocampus_volume', 'Left-Inf-Lat-Vent_volume', 'Left-Lateral-Ventricle_volume', 'Left-non-WM-hypointensities_volume', 'Left-Pallidum_volume', 'Left-Putamen_volume', 'Left-Thalamus-Proper_v

In [4]:
# ============================================================
# Stage 0 — OASIS-3 Column Inventory
# ============================================================

# Columns that identify the participant/session or describe metadata
metadata_cols = [
    "Subject",
    "MR_session",
    "FS_FSDATA ID",
    "Subject_accession",
    "Freesurfer_accession",
    "FS QC Status",
    "version",
    "OASISID",
    "days_from_entry",
    "days_to_visit",
    "GENDER",
]

# Target / clinical variables
target_clinical_cols = [
    "age at visit",
    "AgeatEntry",
    "CDRTOT",
]

# Everything else is a candidate MRI-derived feature
candidate_feature_cols = [
    col for col in oasis.columns
    if col not in metadata_cols + target_clinical_cols
]

print("OASIS-3 column inventory")
print("=" * 50)

print(f"Total columns:              {len(oasis.columns)}")
print(f"Metadata columns:           {len(metadata_cols)}")
print(f"Target/clinical columns:    {len(target_clinical_cols)}")
print(f"Candidate MRI features:     {len(candidate_feature_cols)}")

print("\nMetadata columns:")
print(metadata_cols)

print("\nTarget/clinical columns:")
print(target_clinical_cols)

print("\nFirst 20 candidate MRI features:")
print(candidate_feature_cols[:20])

OASIS-3 column inventory
Total columns:              210
Metadata columns:           11
Target/clinical columns:    3
Candidate MRI features:     196

Metadata columns:
['Subject', 'MR_session', 'FS_FSDATA ID', 'Subject_accession', 'Freesurfer_accession', 'FS QC Status', 'version', 'OASISID', 'days_from_entry', 'days_to_visit', 'GENDER']

Target/clinical columns:
['age at visit', 'AgeatEntry', 'CDRTOT']

First 20 candidate MRI features:
['IntraCranialVol', 'lhCortexVol', 'rhCortexVol', 'CortexVol', 'SubCortGrayVol', 'TotalGrayVol', 'SupraTentorialVol', 'lhCorticalWhiteMatterVol', 'rhCorticalWhiteMatterVol', 'CorticalWhiteMatterVol', '3rd-Ventricle_volume', '4th-Ventricle_volume', '5th-Ventricle_volume', 'Brain-Stem_volume', 'CC_Anterior_volume', 'CC_Central_volume', 'CC_Mid_Anterior_volume', 'CC_Mid_Posterior_volume', 'CC_Posterior_volume', 'CSF_volume']


In [5]:
# ============================================================
# Stage 0 — OASIS-3 Feature Type Inventory
# ============================================================

def classify_feature(name):
    name_lower = name.lower()

    if "thickness" in name_lower:
        return "Cortical thickness"
    elif "surfarea" in name_lower:
        return "Surface area"
    elif "volume" in name_lower or "vol" in name_lower:
        return "Volume"
    elif "numvert" in name_lower:
        return "Vertex count"
    else:
        return "Other morphometric"

feature_inventory = pd.DataFrame({
    "feature": candidate_feature_cols
})

feature_inventory["type"] = feature_inventory["feature"].apply(
    classify_feature
)

print("OASIS-3 MRI Feature Types")
print("=" * 50)

print(
    feature_inventory["type"]
    .value_counts()
    .to_string()
)

print("\nTotal candidate features:",
      len(feature_inventory))

OASIS-3 MRI Feature Types
type
Volume                124
Cortical thickness     68
Vertex count            2
Surface area            2

Total candidate features: 196


In [6]:
# ============================================================
# Stage 0 — Save OASIS-3 Feature Inventory
# ============================================================

FEATURE_INVENTORY_FILE = MAN / "OASIS3_feature_inventory.csv"
FEATURE_LIST_FILE = MAN / "OASIS3_feature_columns.txt"

# Save detailed inventory
feature_inventory.to_csv(
    FEATURE_INVENTORY_FILE,
    index=False
)

# Save feature names only
with open(FEATURE_LIST_FILE, "w") as f:
    for feature in candidate_feature_cols:
        f.write(feature + "\n")

print("OASIS-3 feature inventory saved.")
print(f"Inventory: {FEATURE_INVENTORY_FILE}")
print(f"Feature list: {FEATURE_LIST_FILE}")

print(f"\nSaved {len(candidate_feature_cols)} candidate MRI features.")

OASIS-3 feature inventory saved.
Inventory: /content/drive/MyDrive/ANR_BrainAge/manifests/oasis3/OASIS3_feature_inventory.csv
Feature list: /content/drive/MyDrive/ANR_BrainAge/manifests/oasis3/OASIS3_feature_columns.txt

Saved 196 candidate MRI features.


In [7]:
# ============================================================
# Stage 0 — OASIS-3 Feature Matrix Quality Check
# ============================================================

X_oasis = oasis[candidate_feature_cols]

print("OASIS-3 Feature Matrix Quality Check")
print("=" * 50)

# Data types
non_numeric = X_oasis.select_dtypes(exclude=np.number).columns.tolist()

# Missing values
missing_counts = X_oasis.isna().sum()
features_with_missing = missing_counts[missing_counts > 0]

# Duplicate participants
duplicate_participants = oasis["Subject"].duplicated().sum()

# Duplicate feature names
duplicate_features = (
    pd.Series(candidate_feature_cols)
    .duplicated()
    .sum()
)

print(f"Participants:             {len(oasis)}")
print(f"Features:                 {len(candidate_feature_cols)}")
print(f"Non-numeric features:     {len(non_numeric)}")
print(f"Features with missing:    {len(features_with_missing)}")
print(f"Missing feature values:   {missing_counts.sum()}")
print(f"Duplicate participants:   {duplicate_participants}")
print(f"Duplicate feature names:  {duplicate_features}")

if len(features_with_missing) > 0:
    print("\nFeatures with missing values:")
    print(features_with_missing)

if len(non_numeric) > 0:
    print("\nNon-numeric features:")
    print(non_numeric)

# Hard checks
assert len(non_numeric) == 0
assert duplicate_participants == 0
assert duplicate_features == 0

print("\n✓ OASIS-3 feature matrix passes the basic quality checks.")

OASIS-3 Feature Matrix Quality Check
Participants:             1049
Features:                 196
Non-numeric features:     0
Features with missing:    0
Missing feature values:   0
Duplicate participants:   0
Duplicate feature names:  0

✓ OASIS-3 feature matrix passes the basic quality checks.


---
## Stage 0: OASIS-3 Feature Inventory

The OASIS-3 feature space was characterized and subjected to basic data-quality validation as the first stage of the Build 1 preprocessing pipeline.

The resulting dataset contains **1,049 participants** and **196 MRI-derived morphometric features**, comprising:

- 124 volumetric features
- 68 cortical thickness features
- 2 surface-area features
- 2 vertex-count features

No missing values, duplicate participants, duplicate feature names, or non-numeric feature values were identified in the candidate feature matrix.

The feature inventory and feature-name specification were exported for reproducibility:

- `OASIS3_feature_inventory.csv`
- `OASIS3_feature_columns.txt`

This stage establishes the OASIS-3 feature space for subsequent cross-dataset compatibility assessment with OpenBHB. Model development is intentionally deferred until feature compatibility between the two datasets has been established.